# Implement a Swin Transformer Block (Shifted Window Attention)

**Difficulty**: 🔴 Hard

**Companies**: Microsoft, Meta, ByteDance, Google

---

### Problem Statement

The **Swin Transformer** made pure-attention vision backbones practical by replacing global attention with attention inside **non-overlapping local windows**, which is O(n·w²) and maps perfectly onto dense GPU kernels. But fixed windows have a flaw: tokens near a window edge can never see tokens on the other side.

Swin's fix is **shifted window attention (SW-MSA)**: every other block cyclically shifts the feature map by half a window before partitioning, so new windows straddle the old boundaries and information flows across them. The catch is that shifted windows now mix tokens from distant parts of the image — so each shifted window needs an **attention mask** that only lets tokens attend to tokens that were spatially adjacent *before* the shift.

Your task is to implement a complete Swin block:
1. **Window partition / reverse** — reshape a feature map into per-window token sequences and back.
2. **Window attention** — batched attention within each window, with an optional additive mask.
3. **Shifted-window mask** — the signature piece: build the 0/-inf mask that makes cyclic shifting valid.
4. **`SwinBlock`** — assemble everything: LayerNorm → (shifted) window attention → residual → LayerNorm → MLP → residual.

---

### Requirements

1. **`window_partition(x, window_size)`** — `(B, H, W, C)` → `(B * num_windows, window_size², C)`, non-overlapping windows in row-major order.
2. **`window_reverse(windows, window_size, H, W)`** — the exact inverse of `window_partition`.
3. **`window_attention(q, k, v, mask=None)`** — scaled dot-product attention over `q, k, v` of shape `(B * num_windows, window_size², C)`. `mask` is either `None` or an **additive** mask of shape `(num_windows, window_size², window_size²)` containing `0` (allowed) and `-inf` (blocked).
4. **`create_shifted_window_mask(H, W, window_size, shift_size)`** — returns the additive mask of shape `(num_windows, window_size², window_size²)` for a feature map that will be cyclically shifted by `shift_size` (up and left) before partitioning. Two tokens in the same shifted window may attend to each other **iff they came from the same contiguous region of the pre-shift feature map**.
5. **`SwinBlock(dim, window_size, shift_size, img_size)`** — full block operating on `(B, H, W, C)` with `shift_size=0` (W-MSA) or `shift_size=window_size // 2` (SW-MSA).

---

### Constraints

- ✅ Feature maps are **channels-last**: `(B, H, W, C)`. `H` and `W` are divisible by `window_size` (no padding needed).
- ✅ `window_reverse(window_partition(x, w), w, H, W)` must return `x` exactly.
- ✅ With `shift_size = 0`, no token may attend across a window boundary.
- ✅ With shifting, every token must still attend to **at least one** token (no fully-masked rows — softmax would produce NaNs).
- ✅ The block must preserve the input shape and propagate gradients.
- ❌ Do **not** use `torch.nn.MultiheadAttention`, `timm`, or any existing Swin implementation.
- ℹ️ Single-head attention is fine; multi-head is a bonus (below).

---

<details>
  <summary>💡 Hint</summary>

  - For the shifted mask, don't reason about coordinates *after* shifting. First, label every position in the **pre-shift** feature map by which region it belongs to — slicing at `H - shift` and `W - shift` gives you at most 4 regions. Then apply the *same* shift to the label map that you'll apply to the features (`torch.roll`), and partition the labels into windows. Two tokens may attend iff their labels match.

</details>

---

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Setup: an 8x8 feature map (channels-last), window size 4, shift size 2
torch.manual_seed(42)

batch_size = 2
H, W = 8, 8
dim = 32
window_size = 4
shift_size = window_size // 2

num_windows = (H // window_size) * (W // window_size)
tokens_per_window = window_size * window_size

x = torch.randn(batch_size, H, W, dim)

print(f"Feature map: {x.shape} (B, H, W, C)")
print(f"{num_windows} windows of {tokens_per_window} tokens each")
print(f"Shift size: {shift_size}")

In [ ]:
def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """
    Split a feature map into non-overlapping windows (row-major window order).

    Args:
        x:           (B, H, W, C)
        window_size: side length of each square window

    Returns:
        (B * num_windows, window_size * window_size, C)
    """
    # TODO: Reshape (B, H, W, C) -> (B, H//w, w, W//w, w, C),
    #       permute so window indices come before within-window indices,
    #       then merge batch and window dims.
    pass


def window_reverse(windows: torch.Tensor, window_size: int, H: int, W: int) -> torch.Tensor:
    """
    Merge windows back into a feature map (exact inverse of window_partition).

    Args:
        windows:     (B * num_windows, window_size * window_size, C)
        window_size: side length of each square window
        H, W:        feature map height and width

    Returns:
        (B, H, W, C)
    """
    # TODO: Mirror window_partition: recover B from the leading dim,
    #       reshape, permute back, merge H and W.
    pass


# Quick check: round trip + window contents
windows = window_partition(x, window_size)
print(f"Windows shape: {windows.shape}")  # expect (8, 16, 32)
x_recovered = window_reverse(windows, window_size, H, W)
print(f"Round trip exact: {torch.equal(x_recovered, x)}")
# First window of the first image should be its top-left 4x4 block, flattened
print(f"Window contents correct: "
      f"{torch.equal(windows[0], x[0, :window_size, :window_size].reshape(tokens_per_window, dim))}")

In [ ]:
def window_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                     mask: torch.Tensor = None) -> torch.Tensor:
    """
    Scaled dot-product attention within windows.

    Args:
        q, k, v: (B * num_windows, tokens_per_window, C)
        mask:    None, or additive mask (num_windows, tokens_per_window, tokens_per_window)
                 with 0 = allowed, -inf = blocked

    Returns:
        (B * num_windows, tokens_per_window, C)
    """
    # TODO: Step 1 - scores = q @ k^T / sqrt(C)
    # TODO: Step 2 - if mask is not None, add it to the scores.
    #       The batch dim of scores is B * num_windows but the mask only has
    #       num_windows — view the scores as (B, num_windows, N, N) first.
    # TODO: Step 3 - softmax over the last dim, then weights @ v
    pass


# Quick check: unmasked window attention == full attention per window
wq = torch.randn(batch_size * num_windows, tokens_per_window, dim)
wk = torch.randn(batch_size * num_windows, tokens_per_window, dim)
wv = torch.randn(batch_size * num_windows, tokens_per_window, dim)
out = window_attention(wq, wk, wv)
print(f"Output shape: {out.shape}")
ref = F.softmax(wq @ wk.transpose(-2, -1) / math.sqrt(dim), dim=-1) @ wv
print(f"Matches manual full attention: {torch.allclose(out, ref, atol=1e-6)}")

In [ ]:
def create_shifted_window_mask(H: int, W: int, window_size: int, shift_size: int) -> torch.Tensor:
    """
    Build the additive attention mask for shifted window attention.

    The feature map will be cyclically shifted by shift_size (up and left)
    before partitioning. Within each shifted window, tokens may only attend
    to tokens that belonged to the same contiguous region pre-shift.

    Args:
        H, W:        feature map height and width
        window_size: side length of each square window
        shift_size:  cyclic shift amount

    Returns:
        Float tensor (num_windows, tokens_per_window, tokens_per_window)
        with 0 = allowed, -inf = blocked.
    """
    # TODO: Step 1 - Label each position in the pre-shift map with a region id
    #       (slice rows and columns at H - shift_size / W - shift_size).
    # TODO: Step 2 - Apply the same cyclic shift to the label map.
    # TODO: Step 3 - Partition the labels into windows (you already wrote a
    #       function for this — it works on any channel count).
    # TODO: Step 4 - Within each window, tokens with equal labels get 0,
    #       different labels get -inf.
    pass


# Quick check: visualize the region structure of the bottom-right window
mask = create_shifted_window_mask(H, W, window_size, shift_size)
print(f"Mask shape: {mask.shape}")  # expect (4, 16, 16)
print(f"Allowed tokens per query, last window:\n"
      f"{(mask[-1] == 0).sum(-1).reshape(window_size, window_size)}")

In [ ]:
class SwinBlock(nn.Module):
    """
    One Swin Transformer block: W-MSA (shift_size=0) or SW-MSA (shift_size>0).

    Operates on (B, H, W, C) with H == W == img_size.
    """

    def __init__(self, dim: int, window_size: int, shift_size: int, img_size: int,
                 mlp_ratio: float = 4.0):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
        )

        # TODO: If shift_size > 0, precompute the attention mask for img_size
        #       and store it (register_buffer keeps it on the right device).
        #       Otherwise store None.

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, H, W, C) -> (B, H, W, C)"""
        # TODO: Step 1 - norm1, keep the pre-norm tensor as the residual
        # TODO: Step 2 - if shift_size > 0, cyclically shift the feature map
        #       (torch.roll, negative shift on the H and W dims)
        # TODO: Step 3 - partition into windows, compute q/k/v, apply
        #       window_attention with the precomputed mask, project
        # TODO: Step 4 - reverse the partition, then undo the cyclic shift
        # TODO: Step 5 - add the residual, then x = x + mlp(norm2(x))
        pass


# Quick check: one W-MSA block followed by one SW-MSA block (the classic pair)
block_w = SwinBlock(dim, window_size, shift_size=0, img_size=H)
block_sw = SwinBlock(dim, window_size, shift_size=shift_size, img_size=H)
out = block_sw(block_w(x))
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

In [ ]:
# Validation

# Validation 1: Partition and reverse are exact inverses
print("=== Partition / Reverse Round Trip ===")
windows = window_partition(x, window_size)
assert windows.shape == (batch_size * num_windows, tokens_per_window, dim)
assert torch.equal(window_reverse(windows, window_size, H, W), x)
assert torch.equal(windows[0], x[0, :window_size, :window_size].reshape(tokens_per_window, dim))
print("PASSED: round trip is exact and windows land in the right order.\n")

# Validation 2: With shift_size=0 the mask allows everything within a window
print("=== Zero Shift = No Masking ===")
mask0 = create_shifted_window_mask(H, W, window_size, shift_size=0)
assert mask0.shape == (num_windows, tokens_per_window, tokens_per_window)
assert (mask0 == 0).all(), "shift_size=0 should mask nothing within windows!"
print("PASSED: shift_size=0 masks nothing.\n")

# Validation 3: The shifted mask has the exact expected structure
print("=== Shifted Mask Structure ===")
mask = create_shifted_window_mask(H, W, window_size, shift_size)
allowed = (mask == 0)
# Top-left shifted window lies entirely inside one pre-shift region
assert allowed[0].all(), "window 0 should be fully unmasked!"
# Off-center windows mix two regions: each query sees exactly 8 tokens
assert (allowed[1].sum(-1) == 8).all() and (allowed[2].sum(-1) == 8).all()
# Bottom-right window mixes four regions: each query sees exactly 4 tokens
assert (allowed[3].sum(-1) == 4).all()
# Mask must be symmetric and have no fully-masked rows
assert torch.equal(allowed, allowed.transpose(-2, -1)), "mask must be symmetric!"
assert (allowed.sum(-1) >= 1).all(), "no query may be fully masked!"
print("PASSED: per-window allowed counts are 16 / 8 / 8 / 4, symmetric, no dead rows.\n")

# Validation 4: Shifting actually changes what attention can see
print("=== SW-MSA != W-MSA ===")
torch.manual_seed(0)
blk_a = SwinBlock(dim, window_size, shift_size=0, img_size=H)
torch.manual_seed(0)
blk_b = SwinBlock(dim, window_size, shift_size=shift_size, img_size=H)
assert not torch.allclose(blk_a(x), blk_b(x)), \
    "Identical weights but shifted block should give a different output!"
print("PASSED: shifted and unshifted blocks differ.\n")

# Validation 5: Shape preservation and gradient flow through the block pair
print("=== Forward Shape & Gradients ===")
block_w = SwinBlock(dim, window_size, shift_size=0, img_size=H)
block_sw = SwinBlock(dim, window_size, shift_size=shift_size, img_size=H)
out = block_sw(block_w(x))
assert out.shape == x.shape
out.sum().backward()
assert block_w.qkv.weight.grad is not None and block_sw.qkv.weight.grad is not None
assert torch.isfinite(out).all(), "output contains NaN/Inf — check for fully-masked rows!"
print("PASSED: shape preserved, gradients flow, no NaNs.\n")

# Validation 6: Shifting connects tokens that W-MSA keeps apart
print("=== Shifting Connects Across Old Boundaries ===")
# Pre-shift, tokens (3, 3) and (4, 4) sit in different windows (window 0 and 3),
# so W-MSA can never mix them. After the cyclic shift by 2 they land at shifted
# positions (1, 1) and (2, 2) — both inside shifted window 0, both from the same
# pre-shift region, so SW-MSA must allow the pair.
i, j = 1 * window_size + 1, 2 * window_size + 2  # flattened within-window indices
assert allowed[0][i, j], "shifted window 0 should connect pre-shift tokens (3,3) and (4,4)!"
print("PASSED: SW-MSA bridges the old window boundary.\n")

print("All tests passed!")

---

### Bonus

1. **Relative position bias** — real Swin adds a learned bias to the attention scores based on the *relative* (row, col) offset between query and key inside a window, via a `(2w-1)² × (2w-1)²` table indexed through a cleverly flattened coordinate. This is the other signature Swin mechanism.
2. **Multi-head attention** — split `C` into heads inside `window_attention`.
3. **Patch merging** — between stages, Swin concatenates 2×2 neighborhoods and projects to half the resolution / double the channels, building a hierarchy like a CNN.
4. **Compare** with your `sparse-image-attention` exercise: same asymptotic cost, different access pattern. Why does the disjoint-block version run faster on GPUs in practice?